# Analisi esplorativa dei dati GBIF - biodiversity

...

In [1]:
import os
from pygbif import occurrences as occ
import numpy as np
import pandas as pd

print("test")

test


In [23]:
# Configura le tue credenziali GBIF
# È consigliabile impostarle come variabili d'ambiente nel sistema

import os
import requests
from dotenv import load_dotenv

# Carica le variabili dal file .env nel sistema
load_dotenv()

username = os.getenv("GBIF_USER")
password = os.getenv("GBIF_PWD")
email = os.getenv("GBIF_EMAIL")

os.environ['GBIF_USER'] = username
os.environ['GBIF_PWD'] = password
os.environ['GBIF_EMAIL'] = email

assert username is not None
assert password is not None
assert email is not None

import inspect
print(inspect.signature(occ.download))

# Avvia la richiesta di download (restituisce l'ID della sessione)
download_key = occ.download(
    queries=[
        "taxonKey = 2481947",
        "hasCoordinate = True"],
    format="SIMPLE_CSV",
    user=username,
    pwd=password,
    email=email,
    
)

print(f"Richiesta inviata con successo. Download ID: {download_key}")

# Opzionale: Per scaricare effettivamente il file una volta pronto (esegue un controllo bloccante)
# occ.download_get(download_key, path='./dati_gbif')

(queries, format='SIMPLE_CSV', user=None, pwd=None, email=None, pred_type='and')


INFO:Your download key is 0003826-260928105237408


Richiesta inviata con successo. Download ID: ('0003826-260928105237408', {'creator': 'mattia.jermini', 'notification_address': ['mattia.jermini@gmail.com'], 'sendNotification': True, 'predicate': {'type': 'and', 'predicates': [{'type': 'equals', 'key': 'TAXON_KEY', 'value': '2481947'}, {'type': 'equals', 'key': 'HAS_COORDINATE', 'value': 'True'}]}, 'format': 'SIMPLE_CSV'})


Puoi usare la funzione download_get di pygbif. Questa funzione monitora lo stato del download e, non appena passa da "RUNNING" (in corso) a "SUCCEEDED" (completato), avvia lo scaricamento del file .zip nella cartella che preferisci.


In [25]:
from pygbif import occurrences as occ

download_id = download_key # "0003826-260928105237408"

meta = occ.download_meta(download_id[0])

print(f"Stato attuale del download: {meta['status']}")


Stato attuale del download: SUCCEEDED


## Ispezione iniziale

In [5]:
df.head()

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.columns.to_list()

['country',
 'year',
 'iso_code',
 'population',
 'gdp',
 'cement_co2',
 'cement_co2_per_capita',
 'co2',
 'co2_growth_abs',
 'co2_growth_prct',
 'co2_including_luc',
 'co2_including_luc_growth_abs',
 'co2_including_luc_growth_prct',
 'co2_including_luc_per_capita',
 'co2_including_luc_per_gdp',
 'co2_including_luc_per_unit_energy',
 'co2_per_capita',
 'co2_per_gdp',
 'co2_per_unit_energy',
 'coal_co2',
 'coal_co2_per_capita',
 'consumption_co2',
 'consumption_co2_per_capita',
 'consumption_co2_per_gdp',
 'cumulative_cement_co2',
 'cumulative_co2',
 'cumulative_co2_including_luc',
 'cumulative_coal_co2',
 'cumulative_flaring_co2',
 'cumulative_gas_co2',
 'cumulative_luc_co2',
 'cumulative_oil_co2',
 'cumulative_other_co2',
 'energy_per_capita',
 'energy_per_gdp',
 'flaring_co2',
 'flaring_co2_per_capita',
 'gas_co2',
 'gas_co2_per_capita',
 'ghg_excluding_lucf_per_capita',
 'ghg_per_capita',
 'land_use_change_co2',
 'land_use_change_co2_per_capita',
 'methane',
 'methane_per_capita',
 

In [7]:
df.dtypes

country                            str
year                             int64
iso_code                           str
population                     float64
gdp                            float64
                                ...   
temperature_change_from_n2o    float64
total_ghg                      float64
total_ghg_excluding_lucf       float64
trade_co2                      float64
trade_co2_share                float64
Length: 79, dtype: object

In [8]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
country,50411,254,Afghanistan,275,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,50411.0,NaN,NaN,NaN,1920.349249,65.859123,1750.0,1875.0,1925.0,1975.0,2024.0
iso_code,42480,218,AFG,275,NaN,NaN,NaN,NaN,NaN,NaN,NaN
population,41167.0,NaN,NaN,NaN,60172942.986178,330841153.656435,215.0,327214.0,2291264.0,9986553.0,8161972574.0
gdp,15251.0,NaN,NaN,NaN,330049485119.51355,3086383264219.906738,49980000.0,7874037500.0,27438613750.0,121262743747.5,130112562171125.0
...,...,...,...,...,...,...,...,...,...,...,...
temperature_change_from_n2o,38280.0,NaN,NaN,NaN,0.000509,0.003048,0.0,0.0,0.0,0.0,0.085
total_ghg,38150.0,NaN,NaN,NaN,490.799608,2414.076755,-19.725,1.502,14.6055,76.5085,54433.398
total_ghg_excluding_lucf,37813.0,NaN,NaN,NaN,310.521459,1812.36357,0.0,0.221,2.222,27.863,43714.777
trade_co2,4712.0,NaN,NaN,NaN,-6.986781,259.018184,-2177.807,-2.26225,1.641,11.4255,1768.846


In [9]:
import numpy as np



nullv=df.isnull().sum()
n= df.shape[0]

type(nullv)

for a,b in nullv.items():
    print(a,f"{b}/{n}")


dataset = df

# 1. Ordina il dataset per paese e per anno (fondamentale per le serie storiche)
dataset = dataset.sort_values(by=['country', 'year']).reset_index(drop=True)


# 2. Funzione generica di pulizia esponenziale a prova di nulli iniziali
def pulisci_esponenziale_gruppo(group):
    # Trasforma in logaritmo (gestendo gli zeri)
    log_serie = np.log(group.replace(0, np.nan))
    
    # Interpola i valori intermedi
    log_serie = log_serie.interpolate(method='linear')
    
    # RISOLUZIONE PRIMO VALORE NULLO: bfill copia all'indietro il primo valore valido
    log_serie = log_serie.ffill().bfill()
    
    # Riconverti alla scala originale
    return np.exp(log_serie)

# 3. Applica l'algoritmo separatamente per ogni paese su entrambe le colonne
dataset['population'] = dataset.groupby('country')['population'].transform(pulisci_esponenziale_gruppo)
dataset['gdp'] = dataset.groupby('country')['gdp'].transform(pulisci_esponenziale_gruppo)

# 4. Sicurezza finale per paesi totalmente privi di dati storici
dataset['population'] = dataset['population'].fillna(0)
dataset['gdp'] = dataset['gdp'].fillna(0)

dataset

country 0/50411
year 0/50411
iso_code 7931/50411
population 9244/50411
gdp 35160/50411
cement_co2 21238/50411
cement_co2_per_capita 24763/50411
co2 21027/50411
co2_growth_abs 23195/50411
co2_growth_prct 24172/50411
co2_including_luc 26615/50411
co2_including_luc_growth_abs 26915/50411
co2_including_luc_growth_prct 26915/50411
co2_including_luc_per_capita 26654/50411
co2_including_luc_per_gdp 33621/50411
co2_including_luc_per_unit_energy 40278/50411
co2_per_capita 23902/50411
co2_per_gdp 32883/50411
co2_per_unit_energy 39585/50411
coal_co2 28486/50411
coal_co2_per_capita 29097/50411
consumption_co2 45358/50411
consumption_co2_per_capita 45768/50411
consumption_co2_per_gdp 45963/50411
cumulative_cement_co2 21260/50411
cumulative_co2 22848/50411
cumulative_co2_including_luc 26615/50411
cumulative_coal_co2 28486/50411
cumulative_flaring_co2 34300/50411
cumulative_gas_co2 32264/50411
cumulative_luc_co2 12961/50411
cumulative_oil_co2 24953/50411
cumulative_other_co2 47157/50411
energy_per_ca

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2.802560e+06,9.421400e+09,0.000,0.000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,2.808865e+06,9.421400e+09,0.000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,2.815185e+06,9.421400e+09,0.000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,2.821518e+06,9.421400e+09,0.000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,2.827866e+06,9.421400e+09,0.000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50406,Zimbabwe,2020,ZWE,1.552689e+07,2.317871e+10,0.496,0.032,8.491,-1.776,-17.298,...,NaN,0.106,0.001,0.001,0.002,0.0,24.146,14.463,1.991,23.450
50407,Zimbabwe,2021,ZWE,1.579722e+07,2.514009e+10,0.542,0.034,10.223,1.732,20.398,...,NaN,0.105,0.001,0.001,0.002,0.0,27.907,16.408,2.137,20.899
50408,Zimbabwe,2022,ZWE,1.606906e+07,2.590159e+10,0.387,0.024,12.232,2.009,19.653,...,NaN,0.104,0.001,0.001,0.002,0.0,29.917,18.830,1.380,11.283
50409,Zimbabwe,2023,ZWE,1.634083e+07,2.590159e+10,0.387,0.024,13.443,1.211,9.904,...,NaN,0.103,0.001,0.001,0.002,0.0,31.029,20.318,1.876,13.957


Global temperature (NASA)

In [ ]:
global_temp_nasa = pd.read_csv("https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv")

global_temp_nasa

,,,,,,,,,,,,,,,,,,Land-Ocean: Global Means
Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,J-D,D-N,DJF,MAM,JJA,SON
1880,-.19,-.26,-.10,-.17,-.11,-.22,-.19,-.11,-.15,-.24,-.23,-.18,-.18,***,***,-.13,-.17,-.21
1881,-.20,-.15,.03,.05,.05,-.19,.00,-.04,-.16,-.22,-.19,-.08,-.09,-.10,-.18,.04,-.08,-.19
1882,.16,.13,.04,-.16,-.14,-.23,-.17,-.08,-.15,-.24,-.17,-.37,-.12,-.09,.07,-.09,-.16,-.19
1883,-.30,-.37,-.13,-.19,-.18,-.07,-.08,-.14,-.22,-.11,-.25,-.12,-.18,-.20,-.34,-.16,-.10,-.19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022,.91,.89,1.05,.83,.84,.93,.94,.94,.88,.97,.73,.80,.89,.90,.90,.91,.94,.86
2023,.88,.97,1.23,.99,.94,1.09,1.19,1.19,1.48,1.35,1.40,1.36,1.17,1.12,.88,1.05,1.16,1.41
2024,1.25,1.44,1.40,1.32,1.16,1.23,1.20,1.30,1.21,1.35,1.30,1.27,1.29,1.29,1.35,1.29,1.25,1.29
2025,1.38,1.26,1.37,1.25,1.08,1.07,1.02,1.18,1.25,1.19,1.21,1.06,1.19,1.21,1.30,1.23,1.09,1.22
